In [1]:
!pip install langchain chromadb faiss-cpu langchain_huggingface langchain-community wikipedia sentence-transformers

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

#Wikipedia Retriever

In [2]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
#initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2,lang='en')

#define query
query = 'the geopolitical history of india and pakistan from the perspective of a chinese'

#get relevant wikipedia documents
docs = retriever.invoke(query)

In [4]:
for i,doc in enumerate(docs):
  print(f"\n----Result {i+1} ----")
  print(f"Content:\n{doc.page_content} ")


----Result 1 ----
Content:
The India–Pakistan war of 1965, also known as the second India–Pakistan war, was an armed conflict between Pakistan and India that took place from August 1965 to September 1965.
The conflict began following Pakistan's unsuccessful Operation Gibraltar, which was designed to infiltrate forces into Jammu and Kashmir to precipitate an insurgency against Indian rule. The seventeen day war caused thousands of casualties on both sides and witnessed the largest engagement of armoured vehicles and the largest tank battle since World War II. Hostilities between the two countries ended after a ceasefire was declared through UNSC Resolution 211 following a diplomatic intervention by the Soviet Union and the United States, and the subsequent issuance of the Tashkent Declaration. Much of the war was fought by the countries' land forces in Kashmir and along the border between India and Pakistan. This war saw the largest amassing of troops in Kashmir since the Partition of 

#vector store retriever

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [6]:
from typing_extensions import Doc
#step 1: your source documents
document = [
    Document(page_content='LangChain helps developers build LLM applications easily'),
    Document(page_content='Chroma is a vector database optimized for LLM-based search.'),
    Document(page_content='Embeddings convert text into high-dimensional vectors.'),
    Document(page_content='OpenAI provides powerful embedding models.')
]

In [7]:
!pip install sentence-transformers

In [8]:
#step 2: initialize embedding model
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

#step 3: create chroma vector store in memory
vector_store = Chroma.from_documents(
    documents=document,
    embedding=embedding_model,
    collection_name='my_collection'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
#step4 : convert vectorstore into a retriver
retriever = vector_store.as_retriever(search_kwargs={"k":2})

In [10]:
query = "what is Chroma used for? "
result = retriever.invoke(query)

In [11]:
for i, doc in enumerate(result):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily


In [12]:
results = vector_store.similarity_search(query,k=2)

for i, doc in enumerate(result):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily


#Maximal Marginal Relevance(MMR)

In [13]:
#Sample documents

docs = [
    Document(page_content='LangChain makes it easy to work with LLMs.'),
    Document(page_content='LangChain is used to build LLM based applications.'),
    Document(page_content='Chroma is used to store and search document embeddings'),
    Document(page_content='Embeddings are vector representations of text.'),
    Document(page_content='MMR helps you get diverse results when doing similarity search.'),
    Document(page_content='Langchain supports Chroms, FAISS , PineCone, and more.')
]

In [14]:
from langchain_community.vectorstores import FAISS

#initialize hugging face embeddings
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

#step 2: create the FAISS vector store from documents
vectorestore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [15]:
#enable MMR in the retriever
retriever = vectorestore.as_retriever(
    search_type='mmr',        #this enable MMR
    search_kwargs={"k":3,"lambda_mult":1}  # k = top results, lambda_mult = relevance-diversity balance , 0 -> highest
)

In [16]:
query = "what is langchain?"
results = retriever.invoke(query)

In [17]:
for i, doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)


--- Result 1 ---
Langchain supports Chroms, FAISS , PineCone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
LangChain makes it easy to work with LLMs.


In [19]:
#enable MMR in the retriever
retriever = vectorestore.as_retriever(
    search_type='mmr',        #this enable MMR
    search_kwargs={"k":3,"lambda_mult":0.7}  # k = top results, lambda_mult = relevance-diversity balance , 0 -> highest
)

query = "what is langchain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)


--- Result 1 ---
Langchain supports Chroms, FAISS , PineCone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
LangChain makes it easy to work with LLMs.


#Multi-Query Retriever
### when there is ambiguity in query, generate multiple query from a single query using LLM and then send multiple query to LLM for output and return the output as received from multiple query

In [25]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever

In [30]:
all_docs = [
    Document(page_content='Regular walking boosts heart health and can reduce symptoms of depression',metadata={"source":"H!"}),
    Document(page_content='Consuming leafy greens and fruits helps detox the body and improve longevity',metadata={"source":"H1"}),
    Document(page_content='Deep sleep is crucial for cellular repair and emotional regulation',metadata={"source":"H3"}),
    Document(page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity',metadata={"source":"H4"}),
    Document(page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy',metadata={"source":"H5"}),
    Document(page_content='The solar energy system in modern homes helps balance electricity demand.',metadata={"source":"H1"}),
    Document(page_content='Python balances readability with power,making it popular system design language.',metadata={"source":"H2"}),
    Document(page_content='photosynthesis enables plants to produce energy by converting sunlight.',metadata={"source":"H3"}),
    Document(page_content='The 2022 FIFA world Cup was held in Qatar and drew global energy and excitement',metadata={"source":"H4"}),
    Document(page_content='Black holes bend spacetime and store immense gravitaional energy',metadata={"source":"H5"}),
]

In [31]:
#initializing hugging face embedding model
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

# hugging face chat model
llm = HuggingFaceEndpoint(
    repo_id = "openai/gpt-oss-20b",
    task = "text-generation",
    huggingfacehub_api_token='your_hugging_face_api_key'
)

chat_model = ChatHuggingFace(llm = llm)

In [32]:
#create a FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [33]:
#create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":5})

In [34]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm = chat_model
)

In [35]:
#query
query = "how to improve energy levels and maintain balance"

In [36]:
#retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [37]:
for i,doc in enumerate(similarity_results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

for i,doc in enumerate(multiquery_results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression

--- Result 5 ---
photosynthesis enables plants to produce energy by converting sunlight.

--- Result 6 ---
Mindfulness and controlled breathing 

#contextual compression retriever
###best retriever as it remove unnecessary things from the result

In [41]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [42]:
#recreate the document object from the previous data

from langchain_core.documents import Document

docs = [
    Document(
        page_content="""
        LangChain is a framework for building applications powered by large language models.
        It provides tools for prompt management, retrieval, memory, and agents.
        LangChain supports integrations with OpenAI, HuggingFace, Groq, and many vector databases.
        """,
        metadata={
            "source": "langchain_docs",
            "topic": "framework",
            "level": "beginner",
            "doc_id": 1
        }
    ),
    Document(
        page_content="""
        Retrieval-Augmented Generation (RAG) is a technique that combines document retrieval
        with language model generation. It improves factual accuracy by grounding responses
        in external knowledge sources.
        """,
        metadata={
            "source": "rag_article",
            "topic": "rag",
            "level": "intermediate",
            "doc_id": 2
        }
    ),
    Document(
        page_content="""
        FAISS is a vector database library developed by Facebook AI.
        It enables efficient similarity search using embeddings.
        FAISS is commonly used in RAG pipelines with LangChain.
        """,
        metadata={
            "source": "faiss_blog",
            "topic": "vector_database",
            "level": "intermediate",
            "doc_id": 3
        }
    ),
    Document(
        page_content="""
        ContextualCompressionRetriever reduces the amount of text passed to the language model.
        It uses a compressor model to filter irrelevant parts of retrieved documents
        before sending them to the main LLM.
        """,
        metadata={
            "source": "retriever_notes",
            "topic": "retriever",
            "level": "advanced",
            "doc_id": 4
        }
    ),
    Document(
        page_content="""
        MultiQueryRetriever generates multiple variations of a user query
        to improve retrieval performance and recall in vector databases.
        """,
        metadata={
            "source": "retriever_notes",
            "topic": "retriever",
            "level": "advanced",
            "doc_id": 5
        }
    )
]

In [43]:
# embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# vector store
vectorstore = FAISS.from_documents(docs, embeddings)
base_retriever = vectorstore.as_retriever()

# compressor
llm = chat_model
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

results = compression_retriever.invoke(
    "How does LangChain improve retrieval?"
)

In [44]:
for i,doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)



--- Result 1 ---
It provides tools for prompt management, retrieval, memory, and agents.
LangChain supports integrations with OpenAI, HuggingFace, Groq, and many vector databases.

--- Result 2 ---
ContextualCompressionRetriever reduces the amount of text passed to the language model.  
It uses a compressor model to filter irrelevant parts of retrieved documents  
before sending them to the main LLM.

--- Result 3 ---
MultiQueryRetriever generates multiple variations of a user query
to improve retrieval performance and recall in vector databases.


#Other Retrievals
"""
BM25Retriever
-------------
Keyword-based retriever (no embeddings).
Uses traditional TF-IDF / BM25 scoring.
Good when exact keyword match matters.
Works without vector database.
"""


"""
ParentDocumentRetriever
-----------------------
Retrieves small chunks using embeddings,
but returns the full parent document.
Useful when chunking breaks context
and you want the complete original document.
"""


"""
SelfQueryRetriever
------------------
Uses an LLM to automatically generate
a structured query + metadata filters.
Good when your documents have metadata
and you want intelligent filtering.
"""


"""
TimeWeightedVectorStoreRetriever
---------------------------------
Combines similarity search with recency.
Recent documents get higher weight.
Useful for chat history, news, logs.
"""


"""
MultiVectorRetriever
--------------------
Stores multiple embeddings per document
(e.g., summary embedding + chunk embeddings).
Improves retrieval quality for long documents.
"""


"""
EnsembleRetriever
-----------------
Combines multiple retrievers together
(e.g., BM25 + Vector retriever).
Improves recall and robustness.
"""


"""
ArxivRetriever
--------------
Fetches research papers directly from arXiv.
Useful for academic or research-based queries.
No local vector store required.
"""


Retriever	            Main Idea

BM25	                Keyword search

ParentDocument	      Chunk search → return full doc

SelfQuery	            LLM-generated structured query

TimeWeighted	        Recent docs boosted

MultiVector	          Multiple embeddings per doc

Ensemble	            Combine retrievers

Arxiv	                Search arXiv papers